In [32]:
import pandas as pd
df = pd.read_json('outputs/merged_output_lldatasets_sequential.json')
df = df[df['config'].str.contains("direction=forward", case=False, na=False) & df['config'].str.contains("n_features_to_select=auto", case=False, na=False)]
df.head(10)

,config,dataset,mae_avg,nmae_avg,time(s),folds,features_frequency
1,tol=0.4 direction=forward n_features_to_select...,apache_iteration_30_features,5.966371,0.416030,6836.631763,"[{'mae': 4.826833939878529, 'nmae': 0.26815744...","{'vel_starttime': 2, 'vel_todo': 9, 'no_issues..."
5,tol=0.2 direction=forward n_features_to_select...,apache_iteration_30_features,5.755858,0.403049,8724.152752,"[{'mae': 4.556560216916388, 'nmae': 0.25314223...","{'vel_starttime': 2, 'vel_todo': 9, 'no_issues..."
6,tol=0.15 direction=forward n_features_to_selec...,apache_iteration_30_features,5.615691,0.390431,10174.734339,"[{'mae': 4.556560216916388, 'nmae': 0.25314223...","{'vel_starttime': 3, 'vel_todo': 9, 'no_issues..."
8,tol=0.001 direction=forward n_features_to_sele...,apache_iteration_30_features,5.265374,0.365709,22961.737875,"[{'mae': 4.534888350997866, 'nmae': 0.25193824...","{'vel_starttime': 3, 'vel_todo': 9, 'no_issues..."
11,tol=0.5 direction=forward n_features_to_select...,apache_iteration_30_features,5.966371,0.416030,6638.132060,"[{'mae': 4.826833939878529, 'nmae': 0.26815744...","{'vel_starttime': 2, 'vel_todo': 9, 'no_issues..."
12,tol=1 direction=forward n_features_to_select=auto,apache_iteration_30_features,7.135842,0.501154,4053.118319,"[{'mae': 5.11923236867671, 'nmae': 0.284401798...","{'vel_todo': 9, 'vel_starttime': 1, 'no_issues..."
15,tol=0.3 direction=forward n_features_to_select...,apache_iteration_30_features,5.977187,0.416776,6771.040356,"[{'mae': 4.826833939878529, 'nmae': 0.26815744...","{'vel_starttime': 2, 'vel_todo': 9, 'no_issues..."
17,tol=0.1 direction=forward n_features_to_select...,apache_iteration_30_features,5.475396,0.380370,11965.558422,"[{'mae': 4.556560216916388, 'nmae': 0.25314223...","{'vel_starttime': 3, 'vel_todo': 9, 'no_issues..."
19,tol=0.4 direction=forward n_features_to_select...,jboss_iteration_30_features,3.415193,0.760490,4947.968233,"[{'mae': 4.271912257067286, 'nmae': 0.53398903...","{'no_issue_starttime': 3, 'vel_todo': 10, 'no_..."
26,tol=1 direction=forward n_features_to_select=auto,jboss_iteration_30_features,4.206539,0.945307,3189.082055,"[{'mae': 4.635287963285128, 'nmae': 0.57941099...",{'vel_todo': 10}


In [12]:
import pandas as pd

# Read the CSV file into a DataFrame
df = pd.read_csv('consensus_top_k.csv')
df = df.set_index('algorithm').T
df.reset_index(inplace=True)
df.rename(columns={'index': 'Feature'}, inplace=True)
print()
df .head(10)
df.sort_values(by='Borda count', ascending=True, inplace=True)
df.head(100)



algorithm,Feature,Borda count,Bucket pivot
2,vel_removed,1,1
13,no_issues,2,1
10,vel_added,3,1
1,vel_todo,4,1
0,vel_starttime,5,1
7,no_issue_starttime,6,1
16,no_issue_added,7,2
5,no_comment_min,8,2
6,no_issuetodo,9,2
14,no_issue_removed,10,2


In [36]:
datasets = [
    "apache_iteration_30_features",
    "jboss_iteration_30_features",
    "jira_iteration_30_features",
    "mongodb_iteration_30_features",
    "spring_iteration_30_features",
]

best_configs = []
for dataset in datasets:
    print(f"Best config for dataset: {dataset}:")
    
    # Filter the DataFrame for the current dataset
    filtered_df = df[df['dataset'].str.contains(dataset, case=False, na=False)]
    
    if not filtered_df.empty:
        # Find the index of the best configuration (minimum MAE average)
        best_index = filtered_df['mae_avg'].idxmin()
        
        # Create a new DataFrame for the best configuration using .loc
        best_config = filtered_df.loc[best_index].copy()
        
        print(best_config)
    else:
        print("No entries found for this dataset.")
    
    print("=========================")
    best_configs.append(best_config)


Best config for dataset: apache_iteration_30_features:
config                tol=0.001 direction=forward n_features_to_sele...
dataset                                    apache_iteration_30_features
mae_avg                                                        5.265374
nmae_avg                                                       0.365709
time(s)                                                    22961.737875
folds                 [{'mae': 4.534888350997866, 'nmae': 0.25193824...
features_frequency    {'vel_starttime': 3, 'vel_todo': 9, 'no_issues...
Name: 8, dtype: object
Best config for dataset: jboss_iteration_30_features:
config                tol=0.001 direction=forward n_features_to_sele...
dataset                                     jboss_iteration_30_features
mae_avg                                                        3.068146
nmae_avg                                                       0.677446
time(s)                                                    17626.236678
fold

In [43]:
best_configs_df = pd.DataFrame(best_configs)

def average_features_length(row):
    # 'folds' is a list of dictionaries; each dictionary has a 'features' key with a list of features
    fold_lengths = [len(fold['features']) for fold in row['folds']]
    return sum(fold_lengths) / len(fold_lengths) if fold_lengths else 0

# Apply the function to calculate the average length for each row
best_configs_df['avg_features_length'] = best_configs_df.apply(average_features_length, axis=1)

# Display the first 10 rows with the new 'avg_features_length' column
best_configs_df.head(10)

,config,dataset,mae_avg,nmae_avg,time(s),folds,features_frequency,avg_features_length
8,tol=0.001 direction=forward n_features_to_sele...,apache_iteration_30_features,5.265374,0.365709,22961.737875,"[{'mae': 4.534888350997866, 'nmae': 0.25193824...","{'vel_starttime': 3, 'vel_todo': 9, 'no_issues...",11.5
41,tol=0.001 direction=forward n_features_to_sele...,jboss_iteration_30_features,3.068146,0.677446,17626.236678,"[{'mae': 3.799362223714182, 'nmae': 0.47492027...","{'no_issue_starttime': 3, 'vel_todo': 10, 'no_...",9.5
112,tol=0.001 direction=forward n_features_to_sele...,jira_iteration_30_features,2.229591,0.684537,30477.136216,"[{'mae': 2.5423326993198163, 'nmae': 0.8474442...","{'vel_removed': 1, 'vel_todo': 10, 'no_teammem...",11.9
98,tol=0.001 direction=forward n_features_to_sele...,mongodb_iteration_30_features,4.391634,0.897029,29542.924890,"[{'mae': 5.155800305957981, 'nmae': 0.73654290...","{'no_issue_starttime': 1, 'vel_todo': 10, 'vel...",13.3
107,tol=0.001 direction=forward n_features_to_sele...,spring_iteration_30_features,12.318460,0.373475,25027.565998,"[{'mae': 12.077213376486435, 'nmae': 0.6038606...","{'no_issue_removed': 6, 'vel_todo': 8, 'no_tea...",12.8


In [38]:

existing_df = pd.read_json('outputs/merged_output_all_datasets_sp.json')
# Merge with the existing DataFrame
merged_df = pd.concat([existing_df, best_configs_df], ignore_index=True)

# Display or save the merged DataFrame
merged_df.head(100)

merged_df.to_json('outputs/merged_output_all_datasets_all_models.json', orient='records')